# Camera-domain PPP model comparison
Edit `metric_eval_config.py`, run the evaluator, and inspect the summary and plots.

In [3]:
from pathlib import Path
import h5py

from ppp_offline_metric_eval import load_python_config

cfg = load_python_config("metric_eval_config.py")

for model_name, h5_dir in cfg.H5_DIRS.items():
    print(f"\nModel: {model_name}")
    print(f"Directory: {h5_dir}")

    for path in sorted(Path(h5_dir).glob("rank*.h5")):
        try:
            with h5py.File(path, "r") as h5f:
                keys = list(h5f.keys())
                print(f"  OK:     {path.name}: {len(keys)} samples")
        except Exception as exc:
            print(f"  BROKEN: {path.name}: {exc}")


Model: 1069846_ppp_finals_unet_54k_metric_eval_CamDom
Directory: /mnt/c/Users/uik04723/Repositories/gen_data_metric_finals_cam/1069846_ppp_finals_unet_54k_metric_eval_CamDom/20260728_023617/metric_eval_data/test
  OK:     rank00.h5: 75 samples
  OK:     rank01.h5: 75 samples
  OK:     rank02.h5: 75 samples
  OK:     rank03.h5: 75 samples


In [4]:
from ppp_offline_metric_eval import run_from_config

df_samples, df_summary = run_from_config("metric_eval_config.py")
df_summary

1069846_ppp_finals_unet_54k_metric_eval_CamDom: 300 samples from /mnt/c/Users/uik04723/Repositories/gen_data_metric_finals_cam/1069846_ppp_finals_unet_54k_metric_eval_CamDom/20260728_023617/metric_eval_data/test
  1069846_ppp_finals_unet_54k_metric_eval_CamDom: 1/300
  1069846_ppp_finals_unet_54k_metric_eval_CamDom: 25/300
  1069846_ppp_finals_unet_54k_metric_eval_CamDom: 50/300
  1069846_ppp_finals_unet_54k_metric_eval_CamDom: 75/300
  1069846_ppp_finals_unet_54k_metric_eval_CamDom: 100/300
  1069846_ppp_finals_unet_54k_metric_eval_CamDom: 125/300
  1069846_ppp_finals_unet_54k_metric_eval_CamDom: 150/300
  1069846_ppp_finals_unet_54k_metric_eval_CamDom: 175/300
  1069846_ppp_finals_unet_54k_metric_eval_CamDom: 200/300
  1069846_ppp_finals_unet_54k_metric_eval_CamDom: 225/300
  1069846_ppp_finals_unet_54k_metric_eval_CamDom: 250/300
  1069846_ppp_finals_unet_54k_metric_eval_CamDom: 275/300
  1069846_ppp_finals_unet_54k_metric_eval_CamDom: 300/300

Model comparison summary:
            

,model,n_samples,n_samples_ok,evaluation_ok_fraction,expected_count_mean,gt_count_mean,count_bias,count_mae,count_rmse,count_ratio_mean,...,velocity_rmse_global,velocity_bias_global,velocity_n_points_global,rcs_mae_frame_mean,rcs_rmse_frame_mean,rcs_bias_frame_mean,rcs_mae_global,rcs_rmse_global,rcs_bias_global,rcs_n_points_global
0,1069846_ppp_finals_unet_54k_metric_eval_CamDom,300,300,1.0,165.679799,222.576667,-56.896868,61.449421,74.813649,0.775412,...,1.321607,0.010461,66773,5.631289,7.27477,-0.657096,5.411662,7.194689,-0.511805,66773


In [5]:
main_columns = [
    "model",
    "count_rmse",
    "likelihood_optimal_score_mean",
    "depth_mae_global",
    "velocity_mae_global",
    "rcs_mae_global",
]
display(df_summary[[c for c in main_columns if c in df_summary.columns]].round(4))

,model,count_rmse,likelihood_optimal_score_mean,depth_mae_global,velocity_mae_global,rcs_mae_global
0,1069846_ppp_finals_unet_54k_metric_eval_CamDom,74.8136,7.1458,7.1838,0.4018,5.4117


In [ ]:
from ppp_plotting import plot_expected_count_scatter

plot_expected_count_scatter(
    df_samples,
    output_dir="camera_metric_results/plots",
);

In [ ]:
from ppp_plotting import plot_metric_boxplots

plot_metric_boxplots(
    df_samples,
    [
        "likelihood_optimal_score",
        "depth_mae",
        "velocity_mae",
        "rcs_mae",
    ],
    output_dir="camera_metric_results/plots",
);

In [ ]:
# Verify that every model contains exactly the same sample IDs.
sample_sets = {
    model: set(group["sample_uid"])
    for model, group in df_samples.groupby("model")
}
{model: len(values) for model, values in sample_sets.items()}